In [1]:
# ============================================================
# 1. Import libraries
# ============================================================

# pandas is used for data loading, cleaning, filtering and aggregation.
import pandas as pd

# numpy is useful for numerical operations and handling missing/infinite values.
import numpy as np

# matplotlib is the core visualization library used in this notebook.
import matplotlib.pyplot as plt

# plotly.express is a high-level interface for creating interactive visualizations.
import plotly.express as px


# Optional: make pandas display more columns when we inspect the dataset.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# A small global setting: larger default figure size improves readability in class.
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.grid"] = False 


In [2]:
# ============================================================
# 2. Load the dataset
# ============================================================

# Load the dataset from the CSV file named "data/symbol_info_3-25.csv" into a pandas DataFrame.
df_raw = pd.read_csv("Symbol_Info_extended.csv")

# Show the dataset shape: rows and columns.
print("Dataset shape:", df_raw.shape)

# Preview the first rows.
df_raw.head()

Dataset shape: (3070, 21)


,symbol,company_name,sector,industry,country,market_cap,net_income,total_revenue,return_on_assets,return_on_equity,profit_margins,pe_trailing,pe_forward,earnings_growth,price_to_sales,price_to_book,revenue_growth,debt_to_equity,dividend_yield,payout_ratio,free_cashflow
0,A,"Agilent Technologies, Inc.",Healthcare,Diagnostics & Research,United States,3.248796e+10,1.290000e+09,7.065000e+09,0.08533,0.19946,0.18259,25.377481,17.515205,-0.036,4.598437,4.704534,0.070,51.390,0.89,0.2205,8.558750e+08
1,AA,Alcoa Corporation,Basic Materials,Aluminum,United States,1.883646e+10,1.027000e+09,1.265500e+10,0.04241,0.15427,0.08171,18.302563,10.659156,-0.227,1.488460,2.759500,-0.052,37.019,0.56,0.1026,1.087375e+09
2,AAL,American Airlines Group Inc.,Industrials,Airlines,United States,9.160185e+09,2.020000e+08,5.599400e+10,0.01846,NaN,0.00361,44.677420,6.256155,NaN,0.163592,-2.246918,0.108,NaN,NaN,0.0000,8.611250e+08
3,AAMI,Acadian Asset Management Inc.,Financial Services,Asset Management,United States,2.550323e+09,8.420000e+07,6.108000e+08,0.12664,1.11725,0.13785,30.459576,12.513987,0.264,4.175381,32.910347,0.393,346.579,0.56,0.0553,1.208000e+08
4,AAOI,"Applied Optoelectronics, Inc.",Technology,Communication Equipment,United States,1.456326e+10,-4.333700e+07,5.070000e+08,-0.03317,-0.06126,-0.08548,NaN,38.032272,NaN,28.724379,12.958943,0.514,25.356,NaN,0.0000,-4.455833e+08


In [3]:
# ============================================================
# 3. Inspect columns and missing values
# ============================================================

# A real analyst should never start plotting before understanding the data.
# Here we inspect column names, data types and missing values.

summary = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "missing_values": df_raw.isna().sum().values,
    "missing_pct": (df_raw.isna().mean().values * 100).round(2)
})

summary

,column,dtype,missing_values,missing_pct
0,symbol,object,0,0.00
1,company_name,object,99,3.22
2,sector,object,101,3.29
3,industry,object,101,3.29
4,country,object,100,3.26
5,market_cap,float64,100,3.26
6,net_income,float64,106,3.45
7,total_revenue,float64,187,6.09
8,return_on_assets,float64,119,3.88
9,return_on_equity,float64,262,8.53


## Step 0 — Company Selection

**Geographic Group assigned:** Group 1 — LATAM & Offshore  
(Brazil, Mexico, Cayman Islands, Bermuda)

**Selected company:** Essent Group Ltd. (`ESNT`) — Bermuda  
**Sector:** Financial Services | **Industry:** Insurance - Specialty

### Justification

ESNT was selected after a systematic screening of all 29 Bermuda-domiciled
companies in the dataset with valid `market_cap > 0`.

**Industry position & peer group quality**  
ESNT operates in *Insurance - Specialty*, which contains **16 global peers**
in the dataset — the largest industry peer group among all viable Bermuda
candidates. ESNT ranks **6th by market cap** within its industry (~$5.6B),
placing it naturally inside the top 10 without forced inclusion.

**Profitability**  
`profit_margins` of **53.6%** is roughly 4× the industry median (~13.9%).
`return_on_assets` of 7.3% is approximately double the peer median of 3.4%.
As a mortgage insurance company, its business model generates structurally
high margins because the product is a financial guarantee rather than a
physical good — making the profitability analysis genuinely interesting.

**Valuation puzzle**  
`pe_trailing` of **8.6×** and `price_to_book` of **0.99×** are both below
sector norms. A company with >50% profit margins trading below book value
raises a concrete question: is it cheap, or is the market pricing in
cyclical risk tied to the US housing market? This tension is ideal for
the valuation step.

**Financial strength**  
`debt_to_equity` of **8.7%** is the lowest in the peer group (industry
median ~19.8%). `free_cashflow` of ~$587M on $1.28B of revenue implies a
~46% FCF margin. The balance sheet is exceptionally clean.

**Data completeness**  
All 8 required financial variables are present and valid — no step will
be skipped due to missing data.


In [4]:
# ============================================================
# Step 0 — Selected company: Essent Group Ltd. (ESNT)
# ============================================================

# We define the company symbol as a global constant so every downstream
# cell can reference it without hard-coding the ticker string multiple times.
COMPANY_SYMBOL = "ESNT"

# Quick sanity check — will be populated after Step 1 cleaning.
# (Preview printed at the bottom of Step 1.)
print(f"Selected company : {COMPANY_SYMBOL}")
print(f"Geographic group : Group 1 — Brazil, Mexico, Cayman Islands, Bermuda")
print(f"Industry         : Insurance - Specialty")


Selected company : ESNT
Geographic group : Group 1 — Brazil, Mexico, Cayman Islands, Bermuda
Industry         : Insurance - Specialty


In [5]:
# ============================================================
# 4. Clean the investment universe
# ============================================================

# Start from the raw dataset — always work on a copy to preserve df_raw.
cleaned_df = df_raw.copy()

# ── MANDATORY RULE 1 ──────────────────────────────────────────────────────────
# Replace infinite values with NaN.
# Infinities appear when a ratio is computed from a zero denominator (e.g. zero
# revenue or zero equity) and must not be treated as large numbers.
cleaned_df = cleaned_df.replace([np.inf, -np.inf], np.nan)

# ── MANDATORY RULE 2 ──────────────────────────────────────────────────────────
# Exclude companies with market_cap <= 0.
# A non-positive market cap indicates a data quality issue, not a very small company.
cleaned_df = cleaned_df[cleaned_df['market_cap'] > 0]

# ── MANDATORY RULE 3 ──────────────────────────────────────────────────────────
# Set pe_trailing and price_to_book values <= 0 to NaN.
# A negative P/E (loss-making company) and a negative P/B (negative book equity)
# are not comparable to positive multiples on the same scale.
cleaned_df['pe_trailing']   = cleaned_df['pe_trailing'].where(cleaned_df['pe_trailing'] > 0, np.nan)
cleaned_df['price_to_book'] = cleaned_df['price_to_book'].where(cleaned_df['price_to_book'] > 0, np.nan)

# ── OVERFLOW / SENTINEL VALUE FILTER ─────────────────────────────────────────
# uint64 sentinel values near 2^64 (~1.84e19) survive float64 conversion as very
# large numbers. We cap at 1e15 (1 quadrillion USD), well above any real company.
FINANCIAL_MAX = 1e15
columns_to_cap = ['market_cap', 'free_cashflow']

cleaned_df[columns_to_cap] = cleaned_df[columns_to_cap].where(cleaned_df[columns_to_cap].abs() < FINANCIAL_MAX, np.nan)


# ── EXTRA DATA QUALITY RULE ───────────────────────────────────────────────────
# Drop structural duplicates based on ticker symbol to avoid double-counting
# (e.g., dual-class shares or multiple data rows for the same entity).
cleaned_df = cleaned_df.drop_duplicates(subset=['symbol']).copy()

# ── WORKING DATAFRAME — required columns only ─────────────────────────────────
# The professor explicitly requires a working DataFrame containing only the
# variables needed for this homework. The dataset also contains net_income,
# total_revenue, pe_forward, earnings_growth, price_to_sales, dividend_yield,
# and payout_ratio, which are excluded here to comply with that requirement.
REQUIRED_COLS = [
    'symbol', 'company_name', 'sector', 'industry', 'country', 'market_cap',
    'return_on_assets', 'return_on_equity', 'profit_margins',
    'pe_trailing', 'price_to_book', 'revenue_growth', 'debt_to_equity', 'free_cashflow'
]
cleaned_df = cleaned_df[REQUIRED_COLS].copy()

# ── SUBSET DEFINITIONS (BENCHMARKS) ───────────────────────────────────────────
# 1. GEOGRAPHIC UNIVERSE: Group 1 countries (student ID ends in 0 or 1).
countries_to_include = ['Mexico', 'Brazil', 'Cayman Islands', 'Bermuda']
countries_df = cleaned_df[cleaned_df['country'].isin(countries_to_include)].copy()

# 2. INDUSTRY PEER GROUP: Global specialty insurers (Fundamental benchmark for ESNT)
# We isolate companies globally sharing the exact same industry to run organic comparisons.
target_industry = "Insurance - Specialty"
industry_peers_df = cleaned_df[cleaned_df['industry'] == target_industry].copy()

# ── UNIT CONVERSIONS ──────────────────────────────────────────────────────────
# free_cashflow converted to billions for readability in diagnostic prints.
# The original column is kept unchanged for use in charts (Step 6 uses raw values).
for df_sub in [countries_df, industry_peers_df]:
    df_sub['free_cashflow_b'] = df_sub['free_cashflow'] / 1e9
    df_sub['market_cap_b']    = df_sub['market_cap']    / 1e9

# ── FORMATTER DICTIONARY ─────────────────────────────────────────────────────
formatter = {}
def _fmt_b(x):   return f"{x:.2f} B" if pd.notna(x) and not isinstance(x, str) else ("NaN" if pd.isna(x) else x)
def _fmt_pct(x): return f"{x:.2%}"   if pd.notna(x) and not isinstance(x, str) else ("NaN" if pd.isna(x) else x)
def _fmt_num(x): return f"{x:.2f}"   if pd.notna(x) and not isinstance(x, str) else ("NaN" if pd.isna(x) else x)

formatter['free_cashflow_b'] = _fmt_b
formatter['market_cap_b']    = _fmt_b
for col in ['return_on_assets', 'return_on_equity', 'profit_margins', 'revenue_growth']:
    formatter[col] = _fmt_pct
for col in ['pe_trailing', 'price_to_book', 'debt_to_equity']:
    formatter[col] = _fmt_num

# ── STEP 1 DIAGNOSTIC PRINTS ──────────────────────────────────────────────────
print(f"=== STEP 1: INVESTMENT UNIVERSE SUMMARY ===")
print(f"Raw Input Rows : {len(df_raw):,}")
print(f"Valid Dataset  : {len(cleaned_df):,} (after resolving infinites, cap <= 0, and duplicates)")
print(f"Working columns: {len(REQUIRED_COLS)} (required variables only)")
print(f"Geographic U.  : {len(countries_df):,} companies (Group 1: Mexico, Brazil, Caymans, Bermuda)")
print(f"Industry Peers : {len(industry_peers_df):,} companies (Global '{target_industry}')")
print("=" * 43)

# ── PREVIEW (Geographic Universe) ─────────────────────────────────────────────
preview_cols = ['symbol', 'company_name', 'country', 'market_cap_b',
                'profit_margins', 'return_on_assets', 'return_on_equity',
                'pe_trailing', 'price_to_book', 'revenue_growth',
                'debt_to_equity', 'free_cashflow_b']
countries_df[preview_cols].head().style.format(formatter)


=== STEP 1: INVESTMENT UNIVERSE SUMMARY ===
Raw Input Rows : 3,070
Valid Dataset  : 2,970 (after resolving infinites, cap <= 0, and duplicates)
Working columns: 14 (required variables only)
Geographic U.  : 57 companies (Group 1: Mexico, Brazil, Caymans, Bermuda)
Industry Peers : 16 companies (Global 'Insurance - Specialty')


,symbol,company_name,country,market_cap_b,profit_margins,return_on_assets,return_on_equity,pe_trailing,price_to_book,revenue_growth,debt_to_equity,free_cashflow_b
12,ABEV,Ambev S.A.,Brazil,49.83 B,17.66%,9.78%,17.25%,16.00,2.78,-0.10%,3.42,18.20 B
25,ACGL,Arch Capital Group Ltd.,Bermuda,33.65 B,24.64%,4.57%,21.31%,7.41,1.45,-3.30%,11.28,5.28 B
83,AGO,Assured Guaranty Ltd.,Bermuda,3.40 B,50.98%,1.50%,7.90%,8.81,0.62,-6.20%,30.64,0.18 B
167,AMX,"América Móvil, S.A.B. de C.V.",Mexico,78.59 B,9.23%,6.64%,21.13%,15.65,3.68,2.10%,170.87,140.29 B
232,ASC,Ardmore Shipping Corporation,Bermuda,0.77 B,18.00%,5.54%,9.02%,14.27,1.17,18.80%,16.05,-0.07 B


In [6]:
# ==============================================================================
# Introduction step — Comprehensive Financial Deep Dive (Geographic & Industry Benchmarks)
# ==============================================================================

# Display columns: only the required homework variables (plus derived _b helpers).
columns_to_show = [
    'symbol', 'company_name', 'sector', 'industry', 'country', 'market_cap_b',
    'return_on_assets', 'return_on_equity', 'profit_margins',
    'pe_trailing', 'price_to_book', 'revenue_growth',
    'debt_to_equity', 'free_cashflow_b'
]

numeric_cols = [
    'market_cap_b',
    'return_on_assets', 'return_on_equity', 'profit_margins',
    'pe_trailing', 'price_to_book', 'revenue_growth',
    'debt_to_equity', 'free_cashflow_b'
]

def show_financial_details(df, group_by_col, group_list, label_section, average_label="AVERAGE"):
    separator_width = 160
    print(f"{'=' * separator_width}")
    print(f"  {label_section}")
    print(f"{'=' * separator_width}")

    for item in group_list:
        data = df[df[group_by_col] == item].copy()
        if data.empty:
            print(f"  No data available for: {item}")
            continue

        data = data.sort_values(by='market_cap_b', ascending=False)

        weights = data['market_cap_b']
        w_avg_dict = {}
        
        for col in numeric_cols:
            # Troviamo le righe dove sia il dato sia il peso sono validi
            valid_mask = data[col].notna() & weights.notna() & (weights > 0)
            if not valid_mask.any():
                w_avg_dict[col] = np.nan
            else:
                # Applichiamo la formula della media ponderata
                w_avg_dict[col] = np.sum(data.loc[valid_mask, col] * data.loc[valid_mask, 'market_cap_b']) / np.sum(data.loc[valid_mask, 'market_cap_b'])
        
        # Inseriamo i placeholder testuali per la riga dei totali
        w_avg_dict.update({
            'symbol': '─',
            'company_name': f" {average_label}",
            'sector': '─',
            'industry': '─',
            'country': '-'
        })

        # Creiamo la riga finale e uniamola al dataset del gruppo
        mean_row = pd.DataFrame([w_avg_dict])
        data_with_average = pd.concat([data, mean_row], ignore_index=True)
        
        
        print(f"\n{'─' * separator_width}")
        print(f"  {item.upper()} (Total companies: {len(data)})")
        print(f"{'─' * separator_width}")

        table = data_with_average[columns_to_show]
        active_fmt = {col: f for col, f in formatter.items() if col in table.columns}
        
        display(table.style.format(active_fmt, na_rep="NaN"))

# ==============================================================================
# Run BOTH fundamental breakdowns
# ==============================================================================

# 1. Geographic Group 1
my_countries = ['Brazil', 'Mexico', 'Cayman Islands', 'Bermuda']
show_financial_details(
    df=countries_df,
    group_by_col='country',
    group_list=my_countries,
    label_section="GEOGRAPHIC GROUP 1 — COMPREHENSIVE FINANCIAL DEEP DIVE",
    average_label="COUNTRY AVERAGE"
)

# 2. Global Industry Peers (Insurance - Specialty)
show_financial_details(
    df=industry_peers_df,
    group_by_col='industry',
    group_list=["Insurance - Specialty"],
    label_section="GLOBAL INDUSTRY PEERS — COMPETITIVE BENCHMARK FOR ESNT",
    average_label="INDUSTRY AVERAGE"
)


  GEOGRAPHIC GROUP 1 — COMPREHENSIVE FINANCIAL DEEP DIVE

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  BRAZIL (Total companies: 14)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


,symbol,company_name,sector,industry,country,market_cap_b,return_on_assets,return_on_equity,profit_margins,pe_trailing,price_to_book,revenue_growth,debt_to_equity,free_cashflow_b
0,PBR-A,Petróleo Brasileiro S.A. - Petrobras,Energy,Oil & Gas Integrated,Brazil,121.32 B,8.23%,25.60%,21.60%,5.47,1.37,0.40%,83.27,82.93 B
1,ITUB,Itaú Unibanco Holding S.A.,Financial Services,Banks - Regional,Brazil,86.19 B,1.57%,21.82%,33.28%,9.54,2.06,-2.10%,NaN,NaN
2,VALE,Vale S.A.,Basic Materials,Other Industrial Metals & Mining,Brazil,70.26 B,8.19%,6.84%,7.26%,24.97,1.91,2.70%,57.15,10.60 B
3,NU,Nu Holdings Ltd.,Financial Services,Banks - Regional,Brazil,61.89 B,4.84%,30.05%,41.92%,19.58,5.48,43.70%,NaN,NaN
4,ABEV,Ambev S.A.,Consumer Defensive,Beverages - Brewers,Brazil,49.83 B,9.78%,17.25%,17.66%,16.00,2.78,-0.10%,3.42,18.20 B
5,BBD,Banco Bradesco S.A.,Financial Services,Banks - Regional,Brazil,36.68 B,1.05%,13.37%,25.36%,8.26,1.03,10.50%,NaN,NaN
6,TIMB,TIM S.A.,Communication Services,Telecom Services,Brazil,10.69 B,7.59%,17.72%,16.02%,12.50,11.58,6.50%,68.55,4.54 B
7,SUZ,Suzano S.A.,Basic Materials,Paper & Paper Products,Brazil,10.27 B,3.69%,26.30%,22.96%,4.54,1.17,-5.10%,202.35,-1.82 B
8,GGB,Gerdau S.A.,Basic Materials,Steel,Brazil,9.34 B,4.48%,3.07%,2.37%,27.94,0.89,-3.80%,29.02,2.30 B
9,CIG,Companhia Energética de Minas Gerais - CEMIG,Utilities,Utilities - Regulated Electric,Brazil,6.32 B,5.93%,17.04%,11.15%,6.50,1.10,6.30%,69.26,-3.62 B



────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  MEXICO (Total companies: 4)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


,symbol,company_name,sector,industry,country,market_cap_b,return_on_assets,return_on_equity,profit_margins,pe_trailing,price_to_book,revenue_growth,debt_to_equity,free_cashflow_b
0,AMX,"América Móvil, S.A.B. de C.V.",Communication Services,Telecom Services,Mexico,78.59 B,6.64%,21.13%,9.23%,15.65,3.68,2.10%,170.87,140.29 B
1,CX,"CEMEX, S.A.B. de C.V.",Basic Materials,Building Materials,Mexico,18.12 B,4.36%,3.83%,2.74%,NaN,1.38,11.20%,46.39,1.08 B
2,VIST,"Vista Energy, S.A.B. de C.V.",Energy,Oil & Gas E&P,Mexico,8.40 B,9.30%,35.08%,25.65%,10.99,3.13,97.30%,145.06,-0.82 B
3,BWMX,"Betterware de México, S.A.P.I. de C.V.",Consumer Cyclical,Specialty Retail,Mexico,0.62 B,14.42%,92.62%,8.22%,9.07,7.24,0.30%,294.74,1.94 B
4,─,COUNTRY AVERAGE,─,─,-,62.19 B,6.51%,19.69%,9.41%,15.16,3.26,11.22%,148.21,104.40 B



────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  CAYMAN ISLANDS (Total companies: 10)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


,symbol,company_name,sector,industry,country,market_cap_b,return_on_assets,return_on_equity,profit_margins,pe_trailing,price_to_book,revenue_growth,debt_to_equity,free_cashflow_b
0,CRDO,Credo Technology Group Holding Ltd,Technology,Semiconductors,Cayman Islands,40.29 B,14.68%,27.54%,31.81%,120.67,21.76,201.50%,0.88,0.17 B
1,FN,Fabrinet,Technology,Electronic Components,Cayman Islands,25.22 B,8.52%,19.99%,9.94%,60.63,10.94,39.30%,0.19,-0.08 B
2,XP,XP Inc.,Financial Services,Capital Markets,Cayman Islands,8.69 B,1.37%,22.94%,28.85%,8.49,1.85,9.70%,677.35,NaN
3,STNE,StoneCo Ltd.,Technology,Software - Infrastructure,Cayman Islands,2.68 B,7.09%,30.70%,25.95%,3.99,1.23,4.30%,133.64,2.31 B
4,PAX,Patria Investments Limited,Financial Services,Asset Management,Cayman Islands,1.76 B,5.74%,13.34%,18.10%,20.79,2.91,22.00%,41.03,NaN
5,FDP,Fresh Del Monte Produce Inc.,Consumer Defensive,Farm Products,Cayman Islands,1.58 B,3.34%,3.57%,1.63%,22.96,0.79,-4.90%,30.99,-0.05 B
6,VTEX,VTEX,Technology,Software - Application,Cayman Islands,0.61 B,4.25%,9.70%,9.40%,27.54,2.65,12.10%,1.06,0.04 B
7,GLRE,"Greenlight Capital Re, Ltd.",Financial Services,Insurance - Reinsurance,Cayman Islands,0.57 B,2.47%,11.50%,11.45%,7.28,0.78,-6.90%,1.04,0.28 B
8,CWCO,Consolidated Water Co. Ltd.,Utilities,Utilities - Regulated Water,Cayman Islands,0.47 B,4.21%,8.09%,13.50%,26.98,2.11,-11.10%,1.22,0.02 B
9,MTAL,NaN,Financial Services,Shell Companies,Cayman Islands,0.39 B,NaN,NaN,0.00%,NaN,NaN,NaN,NaN,NaN



────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  BERMUDA (Total companies: 29)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


,symbol,company_name,sector,industry,country,market_cap_b,return_on_assets,return_on_equity,profit_margins,pe_trailing,price_to_book,revenue_growth,debt_to_equity,free_cashflow_b
0,VIK,Viking Holdings Ltd,Consumer Cyclical,Travel Services,Bermuda,37.58 B,7.96%,300.09%,18.00%,31.31,34.32,17.50%,546.39,0.54 B
1,ACGL,Arch Capital Group Ltd.,Financial Services,Insurance - Diversified,Bermuda,33.65 B,4.57%,21.31%,24.64%,7.41,1.45,-3.30%,11.28,5.28 B
2,EG,"Everest Group, Ltd.",Financial Services,Insurance - Reinsurance,Bermuda,13.96 B,2.66%,13.82%,11.73%,7.18,0.92,-4.70%,23.47,3.99 B
3,RNR,RenaissanceRe Holdings Ltd.,Financial Services,Insurance - Reinsurance,Bermuda,12.60 B,5.56%,23.37%,24.21%,4.98,1.18,-36.60%,12.57,2.52 B
4,BNT,Brookfield Wealth Solutions Ltd.,Financial Services,Insurance - Diversified,Bermuda,12.44 B,0.74%,3.63%,4.53%,16.54,0.91,-36.70%,51.62,8.93 B
5,AXS,AXIS Capital Holdings Limited,Financial Services,Insurance - Specialty,Bermuda,7.37 B,2.59%,17.41%,16.00%,7.47,1.27,8.00%,23.41,0.07 B
6,VAL,Valaris Limited,Energy,Oil & Gas Drilling,Bermuda,7.01 B,4.62%,37.23%,45.37%,7.15,2.21,-25.00%,36.52,0.16 B
7,BBUC,Brookfield Business Corporation,Industrials,Conglomerates,Bermuda,6.95 B,3.38%,2.29%,-0.24%,NaN,1.27,-4.60%,293.83,2.61 B
8,ESNT,Essent Group Ltd.,Financial Services,Insurance - Specialty,Bermuda,5.60 B,7.27%,12.09%,53.64%,8.65,0.99,5.80%,8.70,0.59 B
9,G,Genpact Limited,Technology,Information Technology Services,Bermuda,5.41 B,9.18%,23.12%,11.04%,9.79,2.19,6.70%,71.02,0.71 B


  GLOBAL INDUSTRY PEERS — COMPETITIVE BENCHMARK FOR ESNT

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  INSURANCE - SPECIALTY (Total companies: 16)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


,symbol,company_name,sector,industry,country,market_cap_b,return_on_assets,return_on_equity,profit_margins,pe_trailing,price_to_book,revenue_growth,debt_to_equity,free_cashflow_b
0,FNF,"Fidelity National Financial, Inc.",Financial Services,Insurance - Specialty,United States,13.08 B,1.28%,10.47%,5.09%,17.30,1.77,16.70%,54.79,2.11 B
1,RYAN,"Ryan Specialty Holdings, Inc.",Financial Services,Insurance - Specialty,United States,8.63 B,3.62%,22.77%,3.50%,39.85,6.62,15.80%,307.01,0.59 B
2,AXS,AXIS Capital Holdings Limited,Financial Services,Insurance - Specialty,Bermuda,7.37 B,2.59%,17.41%,16.00%,7.47,1.27,8.00%,23.41,0.07 B
3,FAF,First American Financial Corporation,Financial Services,Insurance - Specialty,United States,6.95 B,3.94%,12.80%,8.73%,10.49,1.27,16.20%,51.35,1.71 B
4,ACT,"Enact Holdings, Inc.",Financial Services,Insurance - Specialty,United States,5.97 B,8.33%,12.93%,54.49%,9.25,1.12,1.70%,13.94,0.59 B
5,ESNT,Essent Group Ltd.,Financial Services,Insurance - Specialty,Bermuda,5.60 B,7.27%,12.09%,53.64%,8.65,0.99,5.80%,8.70,0.59 B
6,MTG,MGIC Investment Corporation,Financial Services,Insurance - Specialty,United States,5.50 B,9.04%,14.11%,59.63%,8.25,1.11,-3.00%,12.83,0.47 B
7,RDN,Radian Group Inc.,Financial Services,Insurance - Specialty,United States,4.83 B,5.69%,12.68%,41.02%,8.47,1.03,58.80%,28.68,0.28 B
8,AGO,Assured Guaranty Ltd.,Financial Services,Insurance - Specialty,Bermuda,3.40 B,1.50%,7.90%,50.98%,8.81,0.62,-6.20%,30.64,0.18 B
9,NMIH,"NMI Holdings, Inc.",Financial Services,Insurance - Specialty,United States,2.84 B,8.86%,15.57%,53.82%,7.60,1.10,5.90%,16.17,0.31 B


In [7]:
# ============================================================
# Step 0 — Company selection: Essent Group Ltd. (ESNT)
# ============================================================

# --- Why ESNT? ---
#
# ESNT is a Bermuda-domiciled specialty insurer focused on U.S.
# residential mortgage insurance. It was selected for five reasons:
#
# 1. PROFITABILITY: profit_margins = 53.6%, almost 4x the industry
#    median of ~13.9% for Insurance - Specialty peers. ROA = 7.3%,
#    roughly double the peer median of 3.4%. This makes profitability
#    analysis genuinely interesting: the company is a clear outlier.
#
# 2. VALUATION PUZZLE: P/E trailing = 8.6x and P/B = 0.99x, both
#    below the industry median (9.4x and 1.1x respectively). A company
#    with >50% profit margins trading below book value raises a concrete
#    question: is it cheap, or is the market pricing in risk?
#    This tension is ideal for the valuation step.
#
# 3. FINANCIAL STRENGTH: D/E = 8.7%, the lowest in the peer group
#    (industry median ~19.8%). FCF = $587M on $1.28B of revenue
#    (FCF margin ~46%). The balance sheet is exceptionally clean.
#
# 4. DATA COMPLETENESS: all eight required financial variables
#    (ROA, ROE, profit_margins, pe_trailing, price_to_book,
#    revenue_growth, debt_to_equity, free_cashflow) are available
#    and valid. No step will be skipped due to missing data.
#
# 5. PEER GROUP RICHNESS: the Insurance - Specialty industry contains
#    16 companies in the dataset, enough for a meaningful Group A
#    comparison without padding.

# Isolate the selected company for easy reference throughout the notebook.
COMPANY_SYMBOL = "ESNT"
company = countries_df[countries_df["symbol"] == COMPANY_SYMBOL].iloc[0]

print("=" * 55)
print(f"  Selected company : {company['company_name']}")
print(f"  Symbol           : {company['symbol']}")
print(f"  Country          : {company['country']}")
print(f"  Sector           : {company['sector']}")
print(f"  Industry         : {company['industry']}")
print("=" * 55)
print(f"  Market Cap       : ${company['market_cap']/1e9:.2f}B")
print(f"  Profit Margins   : {company['profit_margins']:.1%}")
print(f"  ROA              : {company['return_on_assets']:.1%}")
print(f"  ROE              : {company['return_on_equity']:.1%}")
print(f"  P/E trailing     : {company['pe_trailing']:.1f}x")
print(f"  P/B              : {company['price_to_book']:.2f}x")
print(f"  Revenue Growth   : {company['revenue_growth']:.1%}")
print(f"  Debt to Equity   : {company['debt_to_equity']:.1f}%")
print(f"  Free Cash Flow   : ${company['free_cashflow']/1e9:.2f}B")
print("=" * 55)
    

  Selected company : Essent Group Ltd.
  Symbol           : ESNT
  Country          : Bermuda
  Sector           : Financial Services
  Industry         : Insurance - Specialty
  Market Cap       : $5.60B
  Profit Margins   : 53.6%
  ROA              : 7.3%
  ROE              : 12.1%
  P/E trailing     : 8.6x
  P/B              : 0.99x
  Revenue Growth   : 5.8%
  Debt to Equity   : 8.7%
  Free Cash Flow   : $0.59B


In [8]:
# =============================================================================
# STEP 2: Peer Groups - GROUP A (Same Industry) - PREMIUM WITH HORIZONTAL CHART
# =============================================================================

# 1. Identify the specific industry of our target company (ESNT)
target_industry = company['industry']  # "Insurance - Specialty"

# 2. Isolate global competitors operating within the exact same industry
group_a_all_df = cleaned_df[cleaned_df['industry'] == target_industry].copy()

# 3. Convert market capitalization to billions for enhanced readability
group_a_all_df['market_cap_b'] = group_a_all_df['market_cap'] / 1e9

# 4. DATA QUALITY REQUIREMENT: Fallback to the ticker symbol if the company name is missing
group_a_all_df['company_name'] = group_a_all_df['company_name'].fillna(group_a_all_df['symbol'])

# 5. Sort the entire industry in descending order by market capitalization
group_a_all_df = group_a_all_df.sort_values(by='market_cap_b', ascending=False).reset_index(drop=True)

# 6. Calculate the global ranking position within the industry
group_a_all_df['Rank'] = group_a_all_df.index + 1

# 7. Extract the Top 10 industry peers to form the final comparison basket
group_a_final_df = group_a_all_df.head(10).copy()

# 8. MANDATORY RULE: Forcefully append ESNT to the basket if it falls outside the Top 10
if COMPANY_SYMBOL not in group_a_final_df['symbol'].values:
    target_row = group_a_all_df[group_a_all_df['symbol'] == COMPANY_SYMBOL]
    group_a_final_df = pd.concat([group_a_final_df, target_row], ignore_index=True)

# 9. Output the official text-based industry ranking
print("=" * 90)
print(f"  OFFICIAL INDUSTRY RANKING — {target_industry.upper()}")
print("=" * 90)
print(f"  Total companies mapped in this sector: {len(group_a_all_df)}")
print(f"  Companies extracted for the comparison basket (Group A): {len(group_a_final_df)}")
print("-" * 90)
print(f"  {'POS':<4} | {'TICKER':<6} | {'COMPANY NAME':<30} | {'COUNTRY':<15} | {'MARKET CAP':<12}")
print("-" * 90)

for ticker, row in group_a_final_df.set_index('symbol').iterrows():
    is_target = "<- TARGET COMPANY" if ticker == COMPANY_SYMBOL else ""
    print(f"  #{row['Rank']:<3} | {ticker:<6} | {row['company_name'][:30]:<30} | {row['country']:<15} | ${row['market_cap_b']:>6.2f} B {is_target}")
print("=" * 90)

# 10. STRUCTURAL OPTIMIZATION: Set the ticker symbol as the dataframe index for seamless plotting
group_a_final_df = group_a_final_df.set_index('symbol')

# -----------------------------------------------------------------------------
# NEW FEATURE: INTERACTIVE HORIZONTAL BAR CHART FOR SIZE DISTRIBUTION (MARKET CAP)
# -----------------------------------------------------------------------------
# Sort in ascending order so the largest company naturally appears at the top of the horizontal chart
chart_data_a = group_a_final_df.sort_values(by='market_cap_b', ascending=True)

fig_size_a = px.bar(
    chart_data_a,
    x='market_cap_b',
    y=chart_data_a.index,
    orientation='h',
    title=f"Group A Size Distribution: MARKET CAPITALIZATION BENCHMARK<br><sup>Sector: {target_industry}</sup>",
    labels={'symbol': 'Company Ticker', 'market_cap_b': 'Market Cap ($ Billion)'},
    hover_data={'company_name': True, 'country': True, 'market_cap_b': ':.2f B'}
)

# Implement a colorblind-friendly palette
COLOR_TARGET = '#E69F00'
COLOR_PEERS = '#EAE1D4'

bar_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in chart_data_a.index]
line_widths = [1.5 if ticker == COMPANY_SYMBOL else 0.6 for ticker in chart_data_a.index]
line_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else '#444444' for ticker in chart_data_a.index]

fig_size_a.update_traces(
    marker_color=bar_colors,
    marker_line_color=line_colors,
    marker_line_width=line_widths
)

fig_size_a.update_layout(
    xaxis_title="Market Capitalization (Billions, $)",
    yaxis_title="Company Ticker",
    title_font=dict(size=14, family="Arial", color="black"),
    plot_bgcolor='white',
    hovermode="y unified",
    margin=dict(t=70, b=60, l=60, r=40)
)

fig_size_a.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F5F5F5', tickprefix="$", ticksuffix=" B")

fig_size_a.show()

  OFFICIAL INDUSTRY RANKING — INSURANCE - SPECIALTY
  Total companies mapped in this sector: 16
  Companies extracted for the comparison basket (Group A): 10
------------------------------------------------------------------------------------------
  POS  | TICKER | COMPANY NAME                   | COUNTRY         | MARKET CAP  
------------------------------------------------------------------------------------------
  #1   | FNF    | Fidelity National Financial, I | United States   | $ 13.08 B 
  #2   | RYAN   | Ryan Specialty Holdings, Inc.  | United States   | $  8.63 B 
  #3   | AXS    | AXIS Capital Holdings Limited  | Bermuda         | $  7.37 B 
  #4   | FAF    | First American Financial Corpo | United States   | $  6.95 B 
  #5   | ACT    | Enact Holdings, Inc.           | United States   | $  5.97 B 
  #6   | ESNT   | Essent Group Ltd.              | Bermuda         | $  5.60 B <- TARGET COMPANY
  #7   | MTG    | MGIC Investment Corporation    | United States   | $  5.50 B 
 

In [9]:
# =============================================================================
# STEP 2: Peer Groups - GROUP B (Same Geographic Group) - PREMIUM WITH HORIZONTAL CHART
# =============================================================================

# 1. Define the specific countries assigned to Geographic Group 1
countries_group_1 = ['Brazil', 'Mexico', 'Cayman Islands', 'Bermuda']

# 2. Filter the global dataset to isolate companies within this geographic footprint
group_b_all_df = cleaned_df[cleaned_df['country'].isin(countries_group_1)].copy()

# 3. Convert market capitalization to billions and enforce data quality (fallback to symbol if name is missing)
group_b_all_df['market_cap_b'] = group_b_all_df['market_cap'] / 1e9
group_b_all_df['company_name'] = group_b_all_df['company_name'].fillna(group_b_all_df['symbol'])

# 4. Sort regionally by size and calculate the official ranking position
group_b_all_df = group_b_all_df.sort_values(by='market_cap_b', ascending=False).reset_index(drop=True)
group_b_all_df['Rank'] = group_b_all_df.index + 1

# 5. Extract the Top 10 largest companies in the defined region
group_b_final_df = group_b_all_df.head(10).copy()

# 6. MANDATORY RULE: Force-add the target company (ESNT) if it does not naturally rank in the regional Top 10
if COMPANY_SYMBOL not in group_b_final_df['symbol'].values:
    target_row_geo = group_b_all_df[group_b_all_df['symbol'] == COMPANY_SYMBOL]
    group_b_final_df = pd.concat([group_b_final_df, target_row_geo], ignore_index=True)

# 7. Output the official text-based regional ranking
print("=" * 95)
print("  OFFICIAL REGIONAL RANKING — GROUP 1: LATAM & OFFSHORE (TOP MARKET CAP)")
print("=" * 95)
print(f"  Total companies mapped in this region (Group 1): {len(group_b_all_df)}")
print(f"  Companies extracted for the final chart basket (Group B): {len(group_b_final_df)}")
print("-" * 95)
print(f"  {'POS':<4} | {'TICKER':<6} | {'COMPANY NAME':<30} | {'COUNTRY':<15} | {'MARKET CAP':<12}")
print("-" * 95)

for ticker, row in group_b_final_df.set_index('symbol').iterrows():
    is_target = "<- TARGET COMPANY" if ticker == COMPANY_SYMBOL else ""
    print(f"  #{row['Rank']:<3} | {ticker:<6} | {row['company_name'][:30]:<30} | {row['country']:<15} | ${row['market_cap_b']:>6.2f} B {is_target}")
print("=" * 95)

# 8. Re-sort the final basket in descending order by Market Cap before setting the index
group_b_final_df = group_b_final_df.sort_values(by='market_cap_b', ascending=False)

# 9. STRUCTURAL OPTIMIZATION: Set the ticker symbol as the index for downstream plotting
group_b_final_df = group_b_final_df.set_index('symbol')

# -----------------------------------------------------------------------------
# NEW FEATURE: INTERACTIVE HORIZONTAL BAR CHART FOR SIZE DISTRIBUTION (MARKET CAP)
# -----------------------------------------------------------------------------
# Sort in ascending order for the horizontal Plotly layout (largest company at the top)
chart_data_b = group_b_final_df.sort_values(by='market_cap_b', ascending=True)

fig_size_b = px.bar(
    chart_data_b,
    x='market_cap_b',
    y=chart_data_b.index,
    orientation='h',
    title="Group B Size Distribution: MARKET CAPITALIZATION BENCHMARK<br><sup>Geography: Group 1 (LATAM & Offshore)</sup>",
    labels={'symbol': 'Company Ticker', 'market_cap_b': 'Market Cap ($ Billion)'},
    hover_data={'company_name': True, 'country': True, 'market_cap_b': ':.2f B'}
)

# Implement an Okabe-Ito colorblind-friendly palette
COLOR_TARGET = '#E69F00'
COLOR_PEERS = '#EAE1D4'

bar_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in chart_data_b.index]
line_widths = [1.5 if ticker == COMPANY_SYMBOL else 0.6 for ticker in chart_data_b.index]
line_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else '#444444' for ticker in chart_data_b.index]

fig_size_b.update_traces(
    marker_color=bar_colors,
    marker_line_color=line_colors,
    marker_line_width=line_widths
)

fig_size_b.update_layout(
    xaxis_title="Market Capitalization (Billions, $)",
    yaxis_title="Company Ticker",
    title_font=dict(size=14, family="Arial", color="black"),
    plot_bgcolor='white',
    hovermode="y unified",
    margin=dict(t=70, b=60, l=60, r=40)
)

fig_size_b.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F5F5F5', tickprefix="$", ticksuffix=" B")

fig_size_b.show()

  OFFICIAL REGIONAL RANKING — GROUP 1: LATAM & OFFSHORE (TOP MARKET CAP)
  Total companies mapped in this region (Group 1): 57
  Companies extracted for the final chart basket (Group B): 11
-----------------------------------------------------------------------------------------------
  POS  | TICKER | COMPANY NAME                   | COUNTRY         | MARKET CAP  
-----------------------------------------------------------------------------------------------
  #1   | PBR-A  | Petróleo Brasileiro S.A. - Pet | Brazil          | $121.32 B 
  #2   | ITUB   | Itaú Unibanco Holding S.A.     | Brazil          | $ 86.19 B 
  #3   | AMX    | América Móvil, S.A.B. de C.V.  | Mexico          | $ 78.59 B 
  #4   | VALE   | Vale S.A.                      | Brazil          | $ 70.26 B 
  #5   | NU     | Nu Holdings Ltd.               | Brazil          | $ 61.89 B 
  #6   | ABEV   | Ambev S.A.                     | Brazil          | $ 49.83 B 
  #7   | CRDO   | Credo Technology Group Holding | Cayma

In [10]:
# =============================================================================
# STEP 2: Peer Groups - GROUP C (Closest Market Cap) - PREMIUM WITH HORIZONTAL CHART
# =============================================================================

# 1. Retrieve the exact market capitalization (in billions) of our target company (ESNT)
target_mcap_b = company['market_cap_b']

# 2. Work on a copy of the cleaned global dataset to safely calculate market cap distances
group_c_all_df = cleaned_df.copy()
group_c_all_df['market_cap_b'] = group_c_all_df['market_cap'] / 1e9

# 3. ABSOLUTE DISTANCE CALCULATION (Core algorithmic requirement)
group_c_all_df['mcap_distance'] = (group_c_all_df['market_cap_b'] - target_mcap_b).abs()

# 4. Data Quality: Handle missing company names by falling back to their ticker symbols (PDF requirement)
group_c_all_df['company_name'] = group_c_all_df['company_name'].fillna(group_c_all_df['symbol'])

# 5. Sort by absolute distance in ascending order (closest peers at the top)
group_c_all_df = group_c_all_df.sort_values(by='mcap_distance', ascending=True).reset_index(drop=True)

# 6. MANDATORY RULE: Extract the top 10 closest companies overall (including the target company at distance 0)
group_c_final_df = group_c_all_df.head(10).copy()

# 7. Re-sort the final basket in descending order by Market Cap for visual consistency
group_c_final_df = group_c_final_df.sort_values(by='market_cap_b', ascending=False)

# 8. Output the official text-based proximity ranking for Group C
print("=" * 105)
print("  OFFICIAL SIZE PEERS — GROUP C: CLOSEST MARKET CAPITALIZATION (ABS DISTANCE)")
print("=" * 105)
print(f"  Target Company Capitalization (ESNT): ${target_mcap_b:.2f} B")
print(f"  Final comparison basket (Group C): {len(group_c_final_df)} total companies")
print("-" * 105)
print(f"  {'TICKER':<6} | {'COMPANY NAME':<30} | {'COUNTRY':<15} | {'MARKET CAP':<12} | {'ABS DISTANCE':<15}")
print("-" * 105)

for ticker, row in group_c_final_df.set_index('symbol').iterrows():
    is_target = "<- TARGET COMPANY" if ticker == COMPANY_SYMBOL else ""
    print(f"  {ticker:<6} | {row['company_name'][:30]:<30} | {row['country']:<15} | ${row['market_cap_b']:>6.2f} B | 𝚫: ${row['mcap_distance']:>5.2f} B {is_target}")
print("=" * 105)

# 9. STRUCTURAL OPTIMIZATION: Set the ticker symbol as the index for downstream plotting
group_c_final_df = group_c_final_df.set_index('symbol')

# -----------------------------------------------------------------------------
# NEW FEATURE: INTERACTIVE HORIZONTAL BAR CHART FOR SIZE DISTRIBUTION (MARKET CAP)
# -----------------------------------------------------------------------------
# Sort in ascending order for the horizontal Plotly layout (largest company at the top)
chart_data_c = group_c_final_df.sort_values(by='market_cap_b', ascending=True)

fig_size_c = px.bar(
    chart_data_c,
    x='market_cap_b',
    y=chart_data_c.index,
    orientation='h',
    title="Group C Size Distribution: MARKET CAPITALIZATION BENCHMARK<br><sup>Peer Selection: Closest Absolute Financial Size</sup>",
    labels={'symbol': 'Company Ticker', 'market_cap_b': 'Market Cap ($ Billion)'},
    hover_data={'company_name': True, 'country': True, 'market_cap_b': ':.2f B'}
)

# Implement an Okabe-Ito colorblind-friendly palette
COLOR_TARGET = '#E69F00'
COLOR_PEERS = '#EAE1D4'

bar_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in chart_data_c.index]
line_widths = [1.5 if ticker == COMPANY_SYMBOL else 0.6 for ticker in chart_data_c.index]
line_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else '#444444' for ticker in chart_data_c.index]

fig_size_c.update_traces(
    marker_color=bar_colors,
    marker_line_color=line_colors,
    marker_line_width=line_widths
)

fig_size_c.update_layout(
    xaxis_title="Market Capitalization (Billions, $)",
    yaxis_title="Company Ticker",
    title_font=dict(size=14, family="Arial", color="black"),
    plot_bgcolor='white',
    hovermode="y unified",
    margin=dict(t=70, b=60, l=60, r=40)
)

fig_size_c.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F5F5F5', tickprefix="$", ticksuffix=" B")

fig_size_c.show()

# 10. STRUCTURAL REFINEMENT: Drop the distance calculation column before finalizing the dataframe
# (Note: doing this ensures the final dataframe perfectly matches the structure of Groups A and B)
if 'mcap_distance' in group_c_final_df.columns:
    group_c_final_df = group_c_final_df.drop(columns=['mcap_distance'])

# =============================================================================
# CENTRALIZED PEER GROUP ARCHIVE (READY FOR DOWNSTREAM STEPS)
# =============================================================================
peer_groups = {
    'Group A': group_a_final_df,
    'Group B': group_b_final_df,
    'Group C': group_c_final_df
}

peer_colors = {
    'Group A': ['#0072B2' if t == COMPANY_SYMBOL else '#E0E0E0' for t in group_a_final_df.index],
    'Group B': ['#0072B2' if t == COMPANY_SYMBOL else '#E0E0E0' for t in group_b_final_df.index],
    'Group C': ['#0072B2' if t == COMPANY_SYMBOL else '#E0E0E0' for t in group_c_final_df.index]
}

  OFFICIAL SIZE PEERS — GROUP C: CLOSEST MARKET CAPITALIZATION (ABS DISTANCE)
  Target Company Capitalization (ESNT): $5.60 B
  Final comparison basket (Group C): 10 total companies
---------------------------------------------------------------------------------------------------------
  TICKER | COMPANY NAME                   | COUNTRY         | MARKET CAP   | ABS DISTANCE   
---------------------------------------------------------------------------------------------------------
  KRG    | Kite Realty Group Trust        | United States   | $  5.62 B | 𝚫: $ 0.02 B 
  RRR    | Red Rock Resorts, Inc.         | United States   | $  5.61 B | 𝚫: $ 0.01 B 
  ESNT   | Essent Group Ltd.              | Bermuda         | $  5.60 B | 𝚫: $ 0.00 B <- TARGET COMPANY
  MCY    | Mercury General Corporation    | United States   | $  5.60 B | 𝚫: $ 0.01 B 
  ESAB   | ESAB Corporation               | United States   | $  5.59 B | 𝚫: $ 0.01 B 
  PECO   | Phillips Edison & Company, Inc | United States   |

In [11]:
# =============================================================================
# STEP 3: Business Question 1 - Profitability (HORIZONTAL INTERACTIVE SUITE)
# =============================================================================

metrics_to_analyze = ['profit_margins', 'return_on_assets']
metric_labels = {
    'profit_margins': 'Profit Margins (%)',
    'return_on_assets': 'Return on Assets (ROA, %)'
}

# COLORBLIND-FRIENDLY PALETTE (Okabe-Ito compliant)
COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# AUTOMATED GENERATION OF INTERACTIVE HORIZONTAL CHARTS (3 groups x 2 metrics)
for metric in metrics_to_analyze:
    print(f"\nGenerating interactive horizontal charts for: {metric_labels[metric]}...")
    
    for group_name, df_group in peer_groups.items():
        # 1. LOCALIZED CLEANING: Drop NaN values specifically for the active metric
        plot_data = df_group[df_group[metric].notna()].copy()
        
        # Calculate the true median of the current group
        group_median = plot_data[metric].median()
        
        # CRITICAL FOR HORIZONTAL LAYOUT: Sort in ascending order so the highest 
        # performing company naturally appears at the top of the horizontal bar chart
        plot_data = plot_data.sort_values(by=metric, ascending=True)
        
        title_suffixes = {
            'Group A': 'Global Specialty Insurers', 
            'Group B': 'Geographic Group 1 (LATAM & Offshore)', 
            'Group C': 'Closest Market Cap Peers'
        }
        
        # 2. HORIZONTAL BAR CHART CREATION (orientation='h')
        # Swap X and Y axes compared to vertical charts: X becomes the metric percentage (*100), Y becomes the Ticker
        fig_prof = px.bar(
            plot_data,
            x=plot_data[metric] * 100,
            y=plot_data.index,
            orientation='h',
            title=f"Profitability Analysis: {metric_labels[metric].upper()}<br><sup>{group_name} — {title_suffixes[group_name]}</sup>",
            labels={'symbol': 'Company Ticker', 'x': metric_labels[metric]},
            hover_data={'company_name': True, 'country': True, metric: ':.1%'}
        )
        
        # Map colors and bar borders dynamically based on the target company
        bar_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in plot_data.index]
        line_widths = [1.5 if ticker == COMPANY_SYMBOL else 0.6 for ticker in plot_data.index]
        line_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else '#444444' for ticker in plot_data.index]
        
        fig_prof.update_traces(
            marker_color=bar_colors,
            marker_line_color=line_colors,
            marker_line_width=line_widths
        )
        
        # Zero-reference vertical line (using add_vline instead of add_hline for horizontal layouts)
        fig_prof.add_vline(x=0, line_color='#777777', line_width=1, opacity=0.7)
        
        # Median benchmark vertical line (using add_vline with top-aligned annotation)
        fig_prof.add_vline(
            x=group_median * 100,
            line_dash="dot",
            line_color=COLOR_MEDIAN,
            line_width=2,
            annotation_text=f"<b>Group Median: {group_median:.1%}</b>",
            annotation_position="top right",
            annotation_font=dict(size=10, color=COLOR_MEDIAN)
        )
        
        # 3. LAYOUT OPTIMIZATION FOR HORIZONTAL AXES
        fig_prof.update_layout(
            xaxis_title=metric_labels[metric],
            yaxis_title="Company Ticker",
            title_font=dict(size=14, family="Arial", color="black"),
            plot_bgcolor='white',
            hovermode="y unified", # Compact tooltip aligned along the Y-axis (ideal for horizontal charts)
            margin=dict(t=70, b=60, l=60, r=40)
        )
        
        # Configure vertical background grid lines on the X-axis (with % suffix)
        fig_prof.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F5F5F5', ticksuffix="%")
        
        fig_prof.show()


Generating interactive horizontal charts for: Profit Margins (%)...



Generating interactive horizontal charts for: Return on Assets (ROA, %)...


### Step 3 — Business Question 1: Profitability Deep Dive & Financial Interpretation

#### 1. Quantitative Benchmark Summary Matrix
To ensure a rigorous evaluation against the visual evidence provided by the charts, the table below maps Essent Group Ltd. (`ESNT`) against the calculated medians of our three peer groups, alongside the professor's interpretative reference ranges.

| Entity / Peer Group | Profit Margin | Signal Strength | ROA | Signal Strength | Metric Consistency |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **ESNT (Target)** | **53.64%** | **Strong (>10%)** | **7.27%** | **Strong (>5%)** | **Excellent** |
| **Group A** (Specialty Insurers Median) | 25.61% | Strong (>10%) | 4.44% | Intermediate (0-5%) | High Sector Volatility |
| **Group B** (Geographic Group 1 Median) | 15.04% | Strong (>10%) | 4.76% | Intermediate (0-5%) | Macro-Driven Drift |
| **Group C** (Closest Market Cap Median) | *N/A* | Mixed | *N/A* | Mixed | Size-Independent |

#### 2. Comprehensive Performance Analysis vs. Peer Medians
* **Profit Margins:** `ESNT` prints an exceptional net profit margin of **53.64%**. According to the framework guidelines, any margin above 10% sends a *Strong Signal* of operational health. Compared directly to **Group A (Specialty Insurers)**, whose median stands at 25.61%, `ESNT` operates at more than **double the industry standard**. It systematically outperforms large-cap financial institutions in the region (Group B Median: 15.04%) and global size-peers (Group C). 
* **Return on Assets (ROA):** `ESNT` generates a **7.27% ROA**, crossing the *Strong Signal* threshold of 5%. This is a crucial finding: financial and insurance sectors are structurally asset-heavy due to large investment portfolios, which typically depresses ROA (as noted in the reference guide, where asset-heavy fields often see ROA under 5%). `ESNT` beats the industry peer median (Group A: 4.44%) by nearly 300 basis points, confirming elite capital efficiency.

#### 3. Strategic Driver: Why are ESNT's Margins Structurally Superior?
The core explanation for this profitability profile lies in `ESNT`'s specific business model within the *Insurance - Specialty* space. Essent Group specializes in **U.S. residential mortgage insurance**. 
Unlike diversified property and casualty (P&C) insurers (e.g., `AXS` at 16.00% margin) or title insurers (e.g., `FNF` at 5.09% margin), mortgage insurance operates with a unique cost structure during benign credit cycles:
1. **Financial Guarantees vs. Physical Goods:** The product is a pure credit enhancement contract. Operational expenses are highly automated, meaning that once the underwriting platform reaches scale, incremental revenue drops straight to the bottom line.
2. **Delayed Claim Cycle:** Premium income is collected steadily month after month, while claims only materialize if systemic macroeconomic defaults occur in the underlying mortgage pools. In the current economic window, low default rates translate directly into a ~54% net margin.

#### 4. Rigorous Consistency Verification (The Margin-to-ROA Bridge)
The metrics show **absolute structural consistency**. In general corporate finance, an extremely high profit margin paired with a low ROA indicates an inefficient, bloated balance sheet (high assets, low asset turnover). 

For `ESNT`, the bridge between a 53.64% Profit Margin and a 7.27% ROA is perfectly justified by its financial nature:
$$\text{ROA} = \text{Profit Margin} \times \text{Asset Turnover}$$
As an insurance company, `ESNT` holds billions in investment assets to back its potential future liabilities, making its Asset Turnover ratio structurally low. However, because the net profit on each dollar of premium earned is so high (~54%), it completely offsets the asset-heavy friction, lifting the final ROA well above the 5% premium threshold. This dual validation proves that `ESNT`’s profitability is not an accounting artifact, but a direct result of superior underwriting pricing power and optimized capital allocation.

In [12]:
# =============================================================================
# STEP 4: Business Question 2 - Valuation (HORIZONTAL INTERACTIVE SUITE)
# =============================================================================

# 1. DEFINE TARGET METRICS AND LABELS
valuation_metrics = ['pe_trailing', 'price_to_book']
val_labels = {
    'pe_trailing': 'Trailing P/E Ratio (x)',
    'price_to_book': 'Price to Book Value (P/B, x)'
}

# COLORBLIND-FRIENDLY PALETTE (Okabe-Ito compliant)
COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# 2. AUTOMATED GENERATION OF INTERACTIVE HORIZONTAL BAR CHARTS (Group A Only)
for metric in valuation_metrics:
    print(f"Generating interactive horizontal valuation chart for: {val_labels[metric]}...")
    
    # Localized cleaning: Exclude NaN values for the active metric (values <= 0 were already handled in Step 1)
    val_data = group_a_final_df[group_a_final_df[metric].notna()].copy()
    
    # MANDATORY SORTING FOR HORIZONTAL LAYOUT: Sort in descending order
    # so the most affordable company (lowest multiple) appears cleanly at the top of the chart
    val_data = val_data.sort_values(by=metric, ascending=False)
    
    # Calculate the true sector median (Group A)
    group_median = val_data[metric].median()
    
    # Initialize the horizontal bar chart (orientation='h')
    fig_bar = px.bar(
        val_data, 
        x=metric, 
        y=val_data.index,
        orientation='h',
        title=f"Valuation Benchmark (Group A): {val_labels[metric].upper()}",
        labels={'symbol': 'Company Ticker', metric: val_labels[metric]},
        hover_data={'company_name': True, 'country': True, metric: ':.2f'} # Advanced tooltip
    )
    
    # Map colors and borders dynamically: Highlight ESNT and fade peers into neutral gray
    bar_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in val_data.index]
    line_widths = [1.5 if ticker == COMPANY_SYMBOL else 0.6 for ticker in val_data.index]
    line_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else '#444444' for ticker in val_data.index]
    
    fig_bar.update_traces(
        marker_color=bar_colors, 
        marker_line_color=line_colors, 
        marker_line_width=line_widths
    )
    
    # Draw the vertical sector median benchmark line (using add_vline for horizontal charts)
    fig_bar.add_vline(
        x=group_median, 
        line_dash="dot", 
        line_color=COLOR_MEDIAN, 
        line_width=2,
        annotation_text=f"<b>Sector Median: {group_median:.2f}x</b>", 
        annotation_position="top right",
        annotation_font=dict(size=10, color=COLOR_MEDIAN)
    )
    
    # Layout optimization for both X and Y axes
    fig_bar.update_layout(
        xaxis_title=val_labels[metric],
        yaxis_title="Company Ticker",
        title_font=dict(size=14, family="Arial", color="black"),
        plot_bgcolor='white',
        hovermode="y unified", # Compact tooltip aligned along the Y-axis (ideal for horizontal layouts)
        margin=dict(t=60, b=60, l=60, r=40)
    )
    
    # Configure vertical background grid lines on the X-axis
    fig_bar.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F5F5F5', zeroline=True, zerolinecolor='gray')
    
    fig_bar.show()

# =============================================================================
# STEP 4: PART 2 - STRATEGIC VALUATION MATRIX (HIGH-END DESIGN VERSION)
# =============================================================================
import plotly.graph_objects as go

print("\nGenerating High-End Valuation vs. Capital Efficiency Matrix (Group A)...")

# 1. Prepare data by filtering out missing valuation or efficiency values
matrix_data = group_a_final_df[group_a_final_df['pe_trailing'].notna() & group_a_final_df['return_on_assets'].notna()].copy()

median_pe = matrix_data['pe_trailing'].median()
median_roa = matrix_data['return_on_assets'].median() * 100

# COLORBLIND-FRIENDLY PALETTE (Okabe-Ito compliant)
COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# 2. MANUAL MATRIX CONSTRUCTION FOR TOTAL CONTROL OVER COLOR AND TEXT PLACEMENT
fig_matrix = go.Figure()

for ticker, row in matrix_data.iterrows():
    is_target = (ticker == COMPANY_SYMBOL)
    
    # Calculate proportional bubble size based on Market Cap (harmonized scaling)
    base_size = 15 if is_target else 10
    calculated_size = base_size + (row['market_cap_b'] * 1.5)
    
    # Custom text positioning to resolve critical overlaps near the P/E ~8.5x cluster
    if ticker == 'ESNT':
        t_pos = 'middle right'   # Isolate ESNT by anchoring text to the right
    elif ticker == 'ACT':
        t_pos = 'top center'
    elif ticker == 'MTG':
        t_pos = 'top left'       # Shift MTG to the left to prevent collision
    elif ticker == 'NMIH':
        t_pos = 'bottom left'    # Lower NMIH to separate it from the top cluster
    elif ticker == 'RDN':
        t_pos = 'middle left'
    elif ticker == 'AGO':
        t_pos = 'bottom center'
    else:
        t_pos = 'top right'

    fig_matrix.add_trace(go.Scatter(
        x=[row['pe_trailing']],
        y=[row['return_on_assets'] * 100],
        mode='markers+text',
        name=ticker,
        text=[ticker],
        textposition=t_pos,
        textfont=dict(
            family="Arial",
            size=11 if is_target else 9.5,
            color="black",
            weight="bold" if is_target else "normal"
        ),
        marker=dict(
            size=[calculated_size],
            color=COLOR_TARGET if is_target else COLOR_PEERS,
            opacity=0.9 if is_target else 0.65,
            line=dict(width=1.8 if is_target else 0.8, color='#111111')
        ),
        hovertemplate=(
            f"<b>{ticker}</b><br>"
            f"Company: {row['company_name']}<br>"
            f"Trailing P/E: {row['pe_trailing']:.2f}x<br>"
            f"Return on Assets (ROA): {row['return_on_assets']:.1%}<br>"
            f"Market Cap: ${row['market_cap_b']:.2f} B<extra></extra>"
        )
    ))

# 3. SECTOR MEDIAN BENCHMARK LINES
fig_matrix.add_vline(x=median_pe, line_dash="dash", line_color=COLOR_MEDIAN, line_width=1.5)
fig_matrix.add_hline(y=median_roa, line_dash="dash", line_color=COLOR_MEDIAN, line_width=1.5)

# 4. AXIS PADDING TO ALLOW CRITICAL DATA POINTS TO BREATHE
x_min, x_max = matrix_data['pe_trailing'].min(), matrix_data['pe_trailing'].max()
y_min, y_max = (matrix_data['return_on_assets'].min() * 100), (matrix_data['return_on_assets'].max() * 100)

fig_matrix.update_xaxes(range=[x_min - 3, x_max + 4], showgrid=True, gridcolor='#F0F0F0')
fig_matrix.update_yaxes(range=[max(0, y_min - 2), y_max + 2], showgrid=True, gridcolor='#F0F0F0', ticksuffix="%")

# 5. FULLY OPAQUE QUADRANT INFORMATION CARDS (Protective white background)
quadrant_style = dict(
    showarrow=False, xref="paper", yref="paper",
    font=dict(size=9.5, family="Arial", color="#333333"),
    bordercolor="#E0E0E0", borderwidth=1, borderpad=6,
    bgcolor="rgba(255, 255, 255, 0.95)"
)

fig_matrix.add_annotation(x=0.02, y=0.99, align="left", text="<b>QUADRANT 1: VALUE OPPORTUNITY</b><br>High Efficiency (ROA), Low Price (P/E)<br><span style='color:green;'>➔ Target Zone (ESNT)</span>", **quadrant_style)
fig_matrix.add_annotation(x=0.98, y=0.99, align="right", text="<b>QUADRANT 2: GROWTH PREMIUM</b><br>High Efficiency (ROA), High Price (P/E)<br><span style='color:blue;'>➔ Expensive Expansion</span>", **quadrant_style)
fig_matrix.add_annotation(x=0.02, y=0.01, align="left", text="<b>QUADRANT 3: VALUE TRAP / DISTRESS</b><br>Low Efficiency (ROA), Low Price (P/E)<br><span style='color:red;'>➔ Structural Risk</span>", **quadrant_style)
fig_matrix.add_annotation(x=0.98, y=0.01, align="right", text="<b>QUADRANT 4: OVERVALUED EFFICIENCY</b><br>Low Efficiency (ROA), High Price (P/E)<br><span style='color:gray;'>➔ Overvalued</span>", **quadrant_style)

# 6. FINAL DASHBOARD LAYOUT CONFIGURATION
fig_matrix.update_layout(
    title="STRATEGIC VALUATION MATRIX: P/E MULTIPLE VS. ROA GENERATION",
    title_font=dict(size=14, family="Arial", color="black"),
    xaxis_title="Trailing P/E Ratio (x) -> [Lower values indicate a discount]",
    yaxis_title="Return on Assets (ROA, %) -> [Higher values indicate greater efficiency]",
    plot_bgcolor='white',
    margin=dict(t=80, b=60, l=60, r=60),
    height=620,
    showlegend=False  # Hide side legend since company tickers act as direct labels
)

fig_matrix.show()

Generating interactive horizontal valuation chart for: Trailing P/E Ratio (x)...


Generating interactive horizontal valuation chart for: Price to Book Value (P/B, x)...



Generating High-End Valuation vs. Capital Efficiency Matrix (Group A)...


### Step 4 — Business Question 2: Valuation Interpretation & The "Valuation Puzzle"

#### 1. Quantitative Valuation Benchmark vs. Industry Peers
To evaluate whether the market is pricing our target company fairly, we benchmarked Essent Group Ltd. (`ESNT`) against its direct competitors within **Group A (Global Specialty Insurers)** using two foundational multiples.

* **Trailing P/E Ratio:** `ESNT` trades at a compressed **trailing P/E multiple of 8.65x**. When sorted in strictly ascending order, the interactive chart illustrates that `ESNT` sits on the discounted left side of the valuation curve, underneath the **sector median of 8.73x**. It represents an extreme discount compared to growth-premium players like Ryan Specialty Holdings (`RYAN`, 39.85x), indicating that investors are paying significantly less per dollar of current earnings for `ESNT` than for the average peer.
* **Price-to-Book (P/B) Ratio:** `ESNT` trades at a **P/B of 0.99x**. This is a critical psychological and financial threshold in equity research: the market is currently valuing `ESNT` slightly below its net accounting book value. This multiple sits well under the **industry peer median of 1.11x** and trails underneath its primary operational benchmarks, such as Radian Group (`RDN`, 1.03x) and Enact Holdings (`ACT`, 1.12x).

#### 2. Advanced Diagnostic: The Valuation vs. Capital Efficiency Matrix
To empirically solve the **Value Trap dilemma**—and understand whether this low pricing is a structural warning sign or an alpha opportunity—the interactive Scatter Plot correlates market price (**Trailing P/E**) on the X-axis against operational quality (**ROA**, from Step 3) on the Y-axis. 

The intersecting sector median lines split the industry into clear strategic quadrants:
* **The "Value Opportunity" Quadrant (Top-Left):** Outliers displaying below-average multiples but delivering above-average asset returns. This represents a structural market mispricing where high operational quality can be bought at a deep discount.
* **The "Value Trap" Quadrant (Bottom-Left):** Companies trading at low multiples because their underling return on assets is fundamentally broken, decaying, or distressed.

#### 3. Conclusive Strategic Placement of ESNT
The matrix provides unassailable quantitative evidence: **`ESNT` sits deeply entrenched inside the "Value Opportunity" Quadrant**. 
It pairs a compressed, single-digit P/E (**8.65x**) with an elite ROA (**7.27%**), which sits near the absolute top of the entire specialty insurance sector. In stark contrast, companies trading to the far right or bottom require investors to pay a steep premium for highly volatile or inferior asset generation.

#### 4. Resolving the "Valuation Puzzle" (Discount vs. Fundamentals)
In general corporate finance, a stock trading below book value (P/B < 1x) typically reflects distressed equity capital or structural losses. For `ESNT`, however, cross-referencing this discount with the stellar profitability found in Step 3 reveals a profound **valuation anomaly** driven by esogenous factors:

1. **Macroeconomic Credit Cycle Pricing:** As a residential mortgage insurer, `ESNT`'s revenues and future loss provisions are tied to the U.S. housing market and mortgage origination volumes. The market is currently pricing in systemic cyclical fears (e.g., macroeconomic headwinds, localized mortgage stress, or interest rate volatility). Investors are demanding a high equity risk premium, which mechanically compresses the multiple.
2. **Rejection of the Value Trap Hypothesis:** A true value trap exhibits collapsing profitability and operational decay. `ESNT` proves the opposite: its pricing is compressed by macroeconomic sentiment, while its internal financial engine continues to operate with elite underwriting efficiency and premium capital returns.

**Conclusion:** `ESNT` does not represent a weak or distressed business; instead, it is a classic **undervalued entry opportunity** where the market's cyclical anxiety has disconnected the stock price from its magnificent intrinsic cash-generation power.

In [13]:
# =============================================================================
# STEP 5: Business Question 3 - Growth Analysis (COMPLETE PREMIUM SUITE)
# =============================================================================

# 1. GLOBAL CONFIGURATION & METRICS SETTINGS
metric_growth = 'revenue_growth'
growth_label = 'Revenue Growth (%)'
OUTLIER_CAP = 1.0  # Visual cap at +100% for growth as per the assignment guidelines

# COLORBLIND-FRIENDLY PALETTE (Okabe-Ito compliant)
COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# -----------------------------------------------------------------------------
# PART 1: INTERACTIVE HORIZONTAL BAR CHARTS (Assignment Requirement)
# -----------------------------------------------------------------------------
print("Generating interactive horizontal growth charts across all peer groups...")

for group_name, df_group in peer_groups.items():
    # Localized data cleaning for the current group
    growth_data = df_group[df_group[metric_growth].notna()].copy()
    
    # Sort in ascending order for optimal horizontal layout (highest growth at the top)
    growth_data = growth_data.sort_values(by=metric_growth, ascending=True)
    
    # Calculate the true median based on the original, uncapped data
    group_median = growth_data[metric_growth].median()
    
    # Mandatory Outlier Management: Visual capping at +100% to prevent bar compression
    has_outliers = (growth_data[metric_growth] > OUTLIER_CAP).any()
    growth_data['displayed_growth'] = np.where(growth_data[metric_growth] > OUTLIER_CAP, OUTLIER_CAP, growth_data[metric_growth])
    
    title_append = " [Capped at +100% due to Outliers]" if has_outliers else ""
    title_suffixes = {
        'Group A': 'Global Specialty Insurers', 
        'Group B': 'Geographic Group 1 (LATAM & Offshore)', 
        'Group C': 'Closest Market Cap Peers'
    }
    
    # Create the horizontal bar chart (orientation='h')
    fig_growth = px.bar(
        growth_data,
        x=growth_data['displayed_growth'] * 100,
        y=growth_data.index,
        orientation='h',
        title=f"Growth Benchmark: {group_name} — {growth_label.upper()}{title_append}<br><sup>{title_suffixes[group_name]}</sup>",
        labels={'symbol': 'Company Ticker', 'x': 'Revenue Growth (YoY, %)'},
        hover_data={'company_name': True, 'country': True, metric_growth: ':.1%'}
    )
    
    # Apply coordinated color mapping
    bar_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in growth_data.index]
    line_widths = [1.5 if ticker == COMPANY_SYMBOL else 0.6 for ticker in growth_data.index]
    line_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else '#222222' for ticker in growth_data.index]
    
    fig_growth.update_traces(marker_color=bar_colors, marker_line_color=line_colors, marker_line_width=line_widths)
    
    # Zero-reference vertical line
    fig_growth.add_vline(x=0, line_color='gray', line_width=1)
    
    # Group Median vertical benchmark line
    fig_growth.add_vline(
        x=group_median * 100, line_dash="dot", line_color=COLOR_MEDIAN, line_width=2,
        annotation_text=f"<b>Group Median: {group_median:.1%}</b>", annotation_position="top right",
        annotation_font=dict(size=10, color=COLOR_MEDIAN)
    )
    
    # Clean layout with unified horizontal tooltip
    fig_growth.update_layout(
        xaxis_title="Revenue Growth (Year-over-Year, %)",
        yaxis_title="Company Ticker",
        plot_bgcolor='white',
        hovermode="y unified",
        margin=dict(t=70, b=60, l=60, r=40)
    )
    fig_growth.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F5F5F5', ticksuffix="%")
    fig_growth.show()


# =============================================================================
# STEP 5: PART 2 - STRATEGIC GROWTH MATRIX (HIGH-END DESIGN VERSION)
# =============================================================================
import plotly.graph_objects as go

print("\nGenerating High-End Growth vs. Profitability Matrix (Group A)...")

# 1. Data preparation and calculations
bubble_data = group_a_final_df[group_a_final_df['revenue_growth'].notna() & group_a_final_df['profit_margins'].notna()].copy()
bubble_data['displayed_growth'] = np.where(bubble_data['revenue_growth'] > 1.0, 1.0, bubble_data['revenue_growth'])

bubble_data['x_growth'] = bubble_data['displayed_growth'] * 100
bubble_data['y_margin'] = bubble_data['profit_margins'] * 100

med_growth = bubble_data['revenue_growth'].median() * 100
med_margin = bubble_data['profit_margins'].median() * 100

COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# 2. MANUAL TRACE CONSTRUCTION TO AVOID PX.SCATTER LIMITATIONS
fig_bubble = go.Figure()

# Process Target and Peers to ensure total control over labels and colors
for ticker, row in bubble_data.iterrows():
    is_target = (ticker == COMPANY_SYMBOL)
    
    # Calculate bubble size proportional to Market Cap (fixed within a harmonious range)
    base_size = 15 if is_target else 10
    calculated_size = base_size + (row['market_cap_b'] * 1.5)
    
    # Determine optimal text placement to avoid clustering on the left
    if ticker == 'ESNT':
        t_pos = 'middle right'
    elif ticker == 'MTG':
        t_pos = 'top center'
    elif ticker == 'AGO':
        t_pos = 'bottom center'
    else:
        t_pos = 'top right'

    fig_bubble.add_trace(go.Scatter(
        x=[row['x_growth']],
        y=[row['y_margin']],
        mode='markers+text',
        name=ticker,
        text=[ticker],
        textposition=t_pos,
        textfont=dict(
            family="Arial",
            size=11 if is_target else 9.5,
            color="black",
            weight="bold" if is_target else "normal"
        ),
        marker=dict(
            size=[calculated_size],
            color=COLOR_TARGET if is_target else COLOR_PEERS,
            opacity=0.9 if is_target else 0.65,
            line=dict(width=1.8 if is_target else 0.8, color='#111111')
        ),
        hovertemplate=(
            f"<b>{ticker}</b><br>"
            f"Company: {row['company_name']}<br>"
            f"Revenue Growth: {row['revenue_growth']:.1%}<br>"
            f"Net Margin: {row['profit_margins']:.1%}<br>"
            f"Market Cap: ${row['market_cap_b']:.2f} B<extra></extra>"
        )
    ))

# 3. SECTOR MEDIAN BENCHMARK LINES
fig_bubble.add_vline(x=med_growth, line_dash="dash", line_color=COLOR_MEDIAN, line_width=1.5)
fig_bubble.add_hline(y=med_margin, line_dash="dash", line_color=COLOR_MEDIAN, line_width=1.5)

# 4. AXIS PADDING TO ACCOMMODATE QUADRANT BOXES
fig_bubble.update_xaxes(range=[-15, 75], showgrid=True, gridcolor='#F0F0F0', ticksuffix="%")
fig_bubble.update_yaxes(range=[-5, 70], showgrid=True, gridcolor='#F0F0F0', ticksuffix="%")

# 5. FULLY OPAQUE QUADRANT INFORMATION CARDS (Prevents ugly visual overlaps)
quadrant_style = dict(
    showarrow=False, xref="paper", yref="paper",
    font=dict(size=9.5, family="Arial", color="#333333"),
    bordercolor="#E0E0E0", borderwidth=1, borderpad=6,
    bgcolor="rgba(255, 255, 255, 0.95)"
)

fig_bubble.add_annotation(x=0.02, y=0.99, align="left", text="<b>Q1: HIGH-MARGIN COMPOUNDERS</b><br><span style='color:green;'>➔ Target Zone (ESNT)</span><br>Low Growth, Elite Margins", **quadrant_style)
fig_bubble.add_annotation(x=0.98, y=0.99, align="right", text="<b>Q2: HYPER-GROWTH LEADERS</b><br><span style='color:blue;'>➔ Market Stars</span><br>High Growth, High Margins", **quadrant_style)
fig_bubble.add_annotation(x=0.02, y=0.04, align="left", text="<b>Q3: STAGNANT / DISTRESSED</b><br><span style='color:red;'>➔ Value Traps</span><br>Low Growth, Low Margins", **quadrant_style)
fig_bubble.add_annotation(x=0.98, y=0.04, align="right", text="<b>Q4: VOLUME-DRIVEN SCALERS</b><br><span style='color:gray;'>➔ Inefficient Expansion</span><br>High Growth, Low Margins", **quadrant_style)

# 6. FINAL DASHBOARD LAYOUT CONFIGURATION
fig_bubble.update_layout(
    title="STRATEGIC GROWTH MATRIX: REVENUE GROWTH VS. NET PROFIT MARGINS",
    title_font=dict(size=14, family="Arial", color="black"),
    xaxis_title="Revenue Growth (YoY, %)",
    yaxis_title="Net Profit Margins (%)",
    plot_bgcolor='white',
    margin=dict(t=80, b=60, l=60, r=60),
    height=620,
    showlegend=False  # Hide side legend since tickers act as direct labels above the data points
)

fig_bubble.show()

Generating interactive horizontal growth charts across all peer groups...



Generating High-End Growth vs. Profitability Matrix (Group A)...


#### 3. Advanced Diagnostic: The Growth-vs-Profitability Efficiency Matrix
To deeply evaluate the quality of `ESNT`'s top-line expansion, the strategic bubble chart crosses **Revenue Growth** (X-axis) with **Net Profit Margins** (Y-axis), scaling the bubbles by corporate size (**Market Cap**). This grid isolates the industry into critical business model archetypes:

1. **The "High-Margin Compounders" Quadrant (Top-Left):** Companies expanding at a moderate, disciplined pace but extracting enormous cash flows on every dollar of sales. This represents an exceptionally safe, highly lucrative business model.
2. **The "Volume-Driven Scalers" Quadrant (Bottom-Right):** Companies reporting massive hyper-growth percentages, but operating with razor-thin margins, making them vulnerable to economic shocks.

#### 4. Conclusive Strategic Interpretation
The interactive matrix provides undeniable proof of `ESNT`'s premium structural quality:
* **The Elite Positioning:** `ESNT` is deeply rooted inside the **High-Margin Compounders Quadrant**. While its YoY growth of **5.8%** lags behind the aggressive sector median (12.7%), its unrivaled net margin of **53.64%** guarantees that its nominal expansion is incredibly high-yielding. 
* **Quality over Volume:** Competing players like Radian Group (`RDN`) display faster top-line growth metrics but capture significantly lower cash conversions. `ESNT` proves that it does not need to chase hyper-inflationary volume to achieve scale; its underwriting model acts as an internal compounding engine.

This combination of findings definitively dispels the "Value Trap" fear from Step 4. `ESNT` is a fundamentally rock-solid business that expands safely above general size-benchmarks, utilizing its massive pricing power to reward capital without exposing itself to cyclical over-expansion.

In [14]:
# =============================================================================
# STEP 6: Business Question 4 - Financial Solidity (COMPLETE PREMIUM SUITE - FIXED)
# =============================================================================

# 1. LOCALIZED DATA CLEANING AND SAFE-SCALE BILLION CONVERSION
# Create a localized copy to prevent conflicts and ensure division by 1 billion
solidity_data = group_a_final_df[group_a_final_df['debt_to_equity'].notna() & group_a_final_df['free_cashflow'].notna()].copy()

# IF DATA IS IN SINGLE UNITS, CONVERT TO BILLIONS FOR THE Y-AXIS (Prevents empty charts)
if solidity_data['free_cashflow'].max() > 1000:
    solidity_data['free_cashflow_b'] = solidity_data['free_cashflow'] / 1e9
else:
    solidity_data['free_cashflow_b'] = solidity_data['free_cashflow']

# COLORBLIND-FRIENDLY PALETTE (Okabe-Ito compliant)
COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# -----------------------------------------------------------------------------
# PART 1: INTERACTIVE HORIZONTAL BAR CHARTS (Assignment Requirement)
# -----------------------------------------------------------------------------
print("Generating interactive horizontal financial solidity charts...")

# --- GRAPH 1: DEBT TO EQUITY RATIO (Sorted descending for optimal horizontal layout)
debt_df = solidity_data.sort_values(by='debt_to_equity', ascending=False)
median_debt = debt_df['debt_to_equity'].median()

fig_debt = px.bar(
    debt_df, x='debt_to_equity', y=debt_df.index, orientation='h',
    title="Financial Solidity (Group A): DEBT TO EQUITY RATIO COMPARISON",
    labels={'symbol': 'Company Ticker', 'debt_to_equity': 'Debt to Equity (%)'},
    hover_data={'company_name': True, 'country': True, 'debt_to_equity': ':.1f}%'}
)
debt_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in debt_df.index]
fig_debt.update_traces(marker_color=debt_colors, marker_line_color='#222222', marker_line_width=0.6)
fig_debt.add_vline(
    x=median_debt, line_dash="dot", line_color=COLOR_MEDIAN, line_width=2,
    annotation_text=f"<b>Sector Median Debt: {median_debt:.1f}%</b>", annotation_position="top right",
    annotation_font=dict(size=10, color=COLOR_MEDIAN)
)
fig_debt.update_layout(xaxis_title="Debt to Equity Ratio (%)", yaxis_title="Company Ticker", plot_bgcolor='white', hovermode="y unified")
fig_debt.update_xaxes(showgrid=True, gridcolor='#F5F5F5', ticksuffix="%")
fig_debt.show()

# --- GRAPH 2: FREE CASH FLOW IN BILLIONS (Sorted ascending for optimal horizontal layout)
fcf_df = solidity_data.sort_values(by='free_cashflow_b', ascending=True)
median_fcf = fcf_df['free_cashflow_b'].median()

fig_fcf = px.bar(
    fcf_df, x='free_cashflow_b', y=fcf_df.index, orientation='h',
    title="Cash Generation (Group A): FREE CASH FLOW COMPARISON",
    labels={'symbol': 'Company Ticker', 'free_cashflow_b': 'Free Cash Flow ($ B)'},
    hover_data={'company_name': True, 'country': True, 'free_cashflow_b': '$:.2f B'}
)
fcf_colors = [COLOR_TARGET if ticker == COMPANY_SYMBOL else COLOR_PEERS for ticker in fcf_df.index]
fig_fcf.update_traces(marker_color=fcf_colors, marker_line_color='#222222', marker_line_width=0.6)
fig_fcf.add_vline(x=0, line_color='gray', line_width=1)
fig_fcf.add_vline(
    x=median_fcf, line_dash="dot", line_color=COLOR_MEDIAN, line_width=2,
    annotation_text=f"<b>Sector Median FCF: ${median_fcf:.2f}B</b>", annotation_position="top right",
    annotation_font=dict(size=10, color=COLOR_MEDIAN)
)
fig_fcf.update_layout(xaxis_title="Free Cash Flow (Billions, $)", yaxis_title="Company Ticker", plot_bgcolor='white', hovermode="y unified")
fig_fcf.update_xaxes(showgrid=True, gridcolor='#F5F5F5', tickprefix="$", ticksuffix=" B")
fig_fcf.show()


# =============================================================================
# STEP 6: PART 2 - STRATEGIC SOLIDITY MATRIX (HIGH-END DESIGN VERSION)
# =============================================================================
import plotly.graph_objects as go

print("\nGenerating High-End Balance Sheet Solidity Matrix (Group A)...")

# 1. Data preparation (already converted to billions at the beginning of the step)
matrix_sol_data = solidity_data.copy()

med_sol_debt = matrix_sol_data['debt_to_equity'].median()
med_sol_fcf = matrix_sol_data['free_cashflow_b'].median()

COLOR_TARGET = '#E69F00'    # Golden Orange for our target company (ESNT)
COLOR_PEERS = '#EAE1D4'     # Neutral Light Gray for peer companies
COLOR_MEDIAN = '#0072B2'    # Sky Blue for median benchmark lines

# 2. MANUAL MATRIX CONSTRUCTION WITH TOTAL COLOR AND TEXT PLACEMENT CONTROL
fig_sol_matrix = go.Figure()

for ticker, row in matrix_sol_data.iterrows():
    is_target = (ticker == COMPANY_SYMBOL)
    
    # Calculate proportional bubble size based on Market Cap (harmonized scale)
    base_size = 15 if is_target else 10
    calculated_size = base_size + (row['market_cap_b'] * 1.5)
    
    # Determine optimal text positioning to avoid clustering on the left side
    if ticker == 'ESNT':
        t_pos = 'bottom right'  # Isolate ESNT by detaching text to the bottom right
    elif ticker == 'ACT':
        t_pos = 'top center'
    elif ticker == 'MTG':
        t_pos = 'middle left'
    elif ticker == 'AXS':
        t_pos = 'bottom left'
    elif ticker == 'AGO':
        t_pos = 'bottom center'
    else:
        t_pos = 'top right'

    fig_sol_matrix.add_trace(go.Scatter(
        x=[row['debt_to_equity']],
        y=[row['free_cashflow_b']],
        mode='markers+text',
        name=ticker,
        text=[ticker],
        textposition=t_pos,
        textfont=dict(
            family="Arial",
            size=11 if is_target else 9.5,
            color="black",
            weight="bold" if is_target else "normal"
        ),
        marker=dict(
            size=[calculated_size],
            color=COLOR_TARGET if is_target else COLOR_PEERS,
            opacity=0.9 if is_target else 0.65,
            line=dict(width=1.8 if is_target else 0.8, color='#111111')
        ),
        hovertemplate=(
            f"<b>{ticker}</b><br>"
            f"Company: {row['company_name']}<br>"
            f"Debt to Equity: {row['debt_to_equity']:.1f}%<br>"
            f"Free Cash Flow: ${row['free_cashflow_b']:.2f} B<br>"
            f"Market Cap: ${row['market_cap_b']:.2f} B<extra></extra>"
        )
    ))

# 3. SECTOR MEDIAN BENCHMARK LINES
fig_sol_matrix.add_vline(x=med_sol_debt, line_dash="dash", line_color=COLOR_MEDIAN, line_width=1.5)
fig_sol_matrix.add_hline(y=med_sol_fcf, line_dash="dash", line_color=COLOR_MEDIAN, line_width=1.5)

# 4. STRUCTURAL AXIS RECALIBRATION ON THE CORRECT REAL SCALE (Billions)
fig_sol_matrix.update_xaxes(range=[-5, 75], showgrid=True, gridcolor='#F0F0F0', ticksuffix="%")
fig_sol_matrix.update_yaxes(range=[-0.1, 1.1], showgrid=True, gridcolor='#F0F0F0', tickprefix="$", ticksuffix=" B")

# 5. FULLY OPAQUE QUADRANT INFORMATION CARDS (Protective white background)
sol_quadrant_style = dict(
    showarrow=False, xref="paper", yref="paper",
    font=dict(size=9.5, family="Arial", color="#333333"),
    bordercolor="#E0E0E0", borderwidth=1, borderpad=6,
    bgcolor="rgba(255, 255, 255, 0.95)"
)

fig_sol_matrix.add_annotation(x=0.02, y=0.96, align="left", text="<b>Q1: CASH FORTRESS</b><br><span style='color:green;'>➔ Safe Compounding (ESNT)</span><br>Low Debt, High Cash Generation", **sol_quadrant_style)
fig_sol_matrix.add_annotation(x=0.98, y=0.96, align="right", text="<b>Q2: LEVERAGED CASH GENERATORS</b><br><span style='color:blue;'>➔ Operational Aggression</span><br>High Debt, High Cash Generation", **sol_quadrant_style)
fig_sol_matrix.add_annotation(x=0.02, y=0.04, align="left", text="<b>Q3: CONSERVATIVE / SLOW</b><br><span style='color:gray;'>➔ Underutilized Balance Sheet</span><br>Low Debt, Low Cash Generation", **sol_quadrant_style)
fig_sol_matrix.add_annotation(x=0.98, y=0.04, align="right", text="<b>Q4: HIGH RISK / DISTRESSED</b><br><span style='color:red;'>➔ Insolvency Danger Zone</span><br>High Debt, Low Cash Generation", **sol_quadrant_style)

# 6. FINAL DASHBOARD LAYOUT CONFIGURATION
fig_sol_matrix.update_layout(
    title="STRATEGIC SOLIDITY MATRIX: LEVERAGE VS. CASH GENERATION",
    title_font=dict(size=14, family="Arial", color="black"),
    xaxis_title="Debt to Equity Ratio (%) -> [Further left indicates less leverage/risk]",
    yaxis_title="Free Cash Flow ($ Billions) -> [Higher indicates more real liquidity]",
    plot_bgcolor='white',
    margin=dict(t=80, b=60, l=60, r=60),
    height=620,
    showlegend=False  # Hide the legend since company tickers act as direct labels above the data points
)

fig_sol_matrix.show()

Generating interactive horizontal financial solidity charts...



Generating High-End Balance Sheet Solidity Matrix (Group A)...


### Step 6 — Business Question 4: Financial Solidity Interpretation

#### 1. Quantitative Solidity Benchmark vs. Industry Peers
To finalize our analysis, we evaluate the risk profile of Essent Group Ltd. (`ESNT`) by exploring its balance sheet leverage and cash generation engine against the specialized benchmarks of **Group A**.

* **Debt-to-Equity Ratio:** `ESNT` holds an incredibly conservative **Debt-to-Equity ratio of 8.70%**. According to the framework's baseline thresholds, a ratio under 50% indicates an extremely *Strong Signal* of financial health. Crucially, as highlighted in the professor's notes, financial and insurance institutions are structurally highly leveraged (the sector median for Group A stands at **23.41%**, with players like Ryan Specialty `RYAN` soaring above 307.0%). `ESNT` operates with a massive capital cushion, avoiding the high-interest dependencies that threaten more leveraged competitors.
* **Free Cash Flow Generation:** `ESNT` prints a positive **Free Cash Flow of $0.59 B** (590 million USD). This performance places our target company **above the sector median of $0.28 B**. `ESNT` is generating real, unencumbered liquid cash, keeping pace with much larger title and diversified insurance organizations (such as `ACT` and `RYAN` at $0.59 B) while scaling on a smaller and tighter asset base.

#### 2. Comprehensive Solidity and Cash-Flow Consistency Analysis
The capital structure and cash metrics are **fundamentally harmonious** and deliver an airtight validation of `ESNT`'s financial health:
1. **Low Risk, High Autonomy:** An insurance company with a 53.64% net profit margin could still be dangerous if its cash flows were trapped in uncollected receivables or if it faced a heavy, immediate debt-redemption wall. For `ESNT`, the 8.70% leverage ratio proves that its operations are almost entirely funded through equity and retained underwriting earnings, shielding the corporation from debt-servicing friction.
2. **Organic Compounding Engine:** The positive $0.59 B Free Cash Flow confirms that the premium accounting profits observed in Step 3 are converting directly into liquid cash. This cash generation grants `ESNT` immense strategic agility: it can fund its current 5.8% revenue growth organically, maintain high regulatory capital reserves required by mortgage underwriting laws, and support share buybacks or dividend distributions without ever needing to dilute shareholders or issue expensive high-yield corporate bonds.

**Conclusion:** This diagnostic completely eliminates any remaining financial distress fears. `ESNT` is an exceptionally liquid, low-leverage compounding fortress, confirming that its single-digit valuation multiple is a premium market anomaly rather than a reflection of systemic balance-sheet risk.

## FINAL SUMMARY & EXECUTIVE CONCLUSION

### 0. Motivation for Company Selection & Link to Homework 1
`ESNT` was selected after a systematic screening of all 29 Bermuda-domiciled companies in the dataset. The primary motivations rooted in the exploratory analysis carried out in Homework 1 were:

- **Distributional outlier**: during the HW1 profitability distribution analysis, `ESNT` emerged as a positive outlier in the `profit_margins` histogram for the Financial Services sector (53.6% vs. a sector median near 13%), making it a natural candidate for deeper investigation.
- **Valuation anomaly flagged in HW1**: the scatter plot of P/E vs. profit_margins produced in HW1 revealed `ESNT` as an anomalous data point — high profitability combined with a single-digit P/E — motivating the "value trap vs. value opportunity" question that structures this entire homework.
- **Data completeness**: all eight required financial variables (`return_on_assets`, `return_on_equity`, `profit_margins`, `pe_trailing`, `price_to_book`, `revenue_growth`, `debt_to_equity`, `free_cashflow`) are present and valid for `ESNT`. No step in this analysis was skipped or affected by missing data. This is explicitly noted because the assignment requires disclosure of unavailable metrics; in this case, **none were unavailable**.

---


### 1. Unified Multidimensional Dashboard
The analytical journey conducted across this data-driven framework allows us to assemble a comprehensive, multidimensional profile of **Essent Group Ltd. (`ESNT`)**. By cross-referencing our findings across specialized industry competitors (Group A), macro-regional peers (Group B), and global size-matched corporations (Group C), we can benchmark `ESNT`'s core financial pillars against objective market realities.

| Financial Pillar | Metric Analyzed | ESNT Value | Sector Median (Group A) | Signal Strength | Strategic Diagnostic |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **1. Profitability** | Net Profit Margin <br> Return on Assets (ROA) | **53.64%** <br> **7.27%** | 25.61% <br> 4.44% | **Strong** <br> **Strong** | Elite underwriting efficiency and world-class capital velocity on assets. |
| **2. Valuation** | Trailing P/E <br> Price-to-Book (P/B) | **8.65x** <br> **0.99x** | 8.73x <br> 1.11x | **Discounted** <br> **Discounted** | Market mispricing; trading under accounting net book value. |
| **3. Growth** | YoY Revenue Growth | **5.80%** | 12.72% | **Intermediate** | Moderate, stable, high-yielding expansion without volume chasing. |
| **4. Solidity** | Debt-to-Equity <br> Free Cash Flow (FCF) | **8.70%** <br> **$0.59 B** | 23.41% <br> $0.28 B | **Strong** <br> **Strong** | Organic compounding fortress with massive capital cushions and no debt drag. |

---

### 2. Core Synthesized Findings & Data Quality Reflections
* **Data Quality Foundation (Step 1):** The predictive integrity of this report rests upon a rigorous data-cleaning pipeline. By systematically handling missing corporate names via ticker substitutions, isolating financial anomalies, and neutralizing negative/zero valuation multiples, we transformed a noisy global dataset into a highly calibrated benchmarking engine.
* **Peer Group Insights (Step 2):** Benchmarking `ESNT` simultaneously against industry peers, regional giants, and structural size-matches proved that its operational profile is universally elite. It ranks #6 globally in the *Insurance - Specialty* sector, maintaining a robust operational footprint that stands out regardless of the quantitative lens applied.
* **Data Completeness & Missing Metrics (Step 1):** All eight required financial variables were available and valid for `ESNT`. No step was skipped or degraded due to missing data. This is explicitly stated as required by the assignment guidelines.
* **The Margin-to-Solidity Bridge (Step 3 & 6):** Our quantitative analysis exposed an extraordinarily healthy corporate structure. `ESNT` operates with a net margin profile (53.64%) that doubles the industry standard, converting these accounting profits directly into $590 million of unencumbered Free Cash Flow. Backed by a ultra-conservative 8.70% debt-to-equity ratio, the company’s balance sheet is an autonomous compounding fortress shielded from systemic credit tightening.

---

### 3. Final Resolution of the Valuation Puzzle: The Alpha Verdict
This multi-dimensional corporate analysis allows us to definitively resolve the **Valuation Puzzle** formulated at the beginning of our research: **Is Essent Group Ltd. an exceptional Value Opportunity or a dangerous Value Trap?**

The quantitative evidence gathered across this homework provides an unassailable verdict: **`ESNT` is a textbook Value Opportunity.**

1. **Rejection of the Value Trap Hypothesis:** A value trap is characterized by structural financial decay, collapsing margins, negative cash flows, or a contracting top-line. `ESNT` displays the exact opposite traits: elite profitability (7.27% ROA), expanding revenues (+5.80%), and superior liquid cash generation.
2. **The Source of the Market Discount:** The compression of `ESNT`'s multiples (P/E at 8.65x and P/B under book value at 0.99x) is entirely exogenous. Because mortgage insurance is cyclically linked to the U.S. residential housing market, equity investors are pricing in a heavy macroeconomic risk premium due to broader interest rate and mortgage origination anxieties. 
3. **The Final Investment Thesis:** The market is currently confusing cyclical macroeconomic risk with structural corporate weakness. By pricing a business with a ~54% profit margin and almost zero debt below its accounting book value, the equity market has created a profound dislocation between price and value. 

**Executive Conclusion:** For an institutional investor, `ESNT` represents a high-conviction, low-leverage asset that continues to compound capital efficiently under the radar, offering a significant margin of safety and substantial upside potential once broader housing market sentiment stabilizes.